In [ ]:
# ============================================================
# Cell 0: 检查 GPU 是否可用
# ============================================================
# 大模型示例最常见的失败原因不是代码错，而是没切到 GPU 运行时。
# 先一眼确认 GPU 在线，再继续后面的步骤，避免白白下载几 GB 权重。
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    # 显存以 GB 为单位打印，T4 应显示约 14.7 GB
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total VRAM: {total_mem:.1f} GB")
else:
    # 切换方式：菜单 → 代码执行程序 → 更改运行时类型 → 硬件加速器 → T4 GPU
    print("⚠️ 当前没有 GPU，请切到 GPU 运行时再继续")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装/升级依赖库
# ============================================================
# 重要：%%capture 是单元格魔法命令，必须是 cell 的第一行（前面不能有任何内容，包括注释）
# 否则 Jupyter / VS Code 会将其当作 line magic 解析，报错：
#   UsageError: Line magic function `%%capture` not found.
# 它的作用：捕获整个 cell 的输出，让 pip install 几十行的安装日志不显示在屏幕上
# ! 前缀让 IPython 把这一行交给系统 shell 执行（而非 Python 解释器）
# -q 表示 quiet 模式，进一步减少 pip 的输出噪音
# -U 表示 upgrade，如果已装则升级到最新版
# transformers>=4.51:  Qwen3 系列要求至少 4.51，否则 chat template 不识别 enable_thinking
# accelerate:           分布式/混合精度加速，加载大模型时几乎是必需依赖
# bitsandbytes:         提供 8-bit / 4-bit 量化能力，让大模型在小显存上跑起来
!pip install -q -U "transformers>=4.51" accelerate bitsandbytes

In [ ]:
# ============================================================
# Cell 2: 加载模型与分词器（采用 4-bit 量化）
# ============================================================
# AutoModelForCausalLM: "自动选择 CausalLM 类"的工厂类
#   CausalLM = Causal Language Model，因果语言模型，即 GPT 这类自回归生成模型
# AutoTokenizer:        根据模型 ID 自动选择对应的分词器实现
# BitsAndBytesConfig:   配置 bitsandbytes 的量化参数
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Hugging Face Hub 模型 ID，格式 "组织名/模型名"
# Qwen3 系列把 base 与 chat 合并到同一仓库，仓库名不再带 "-Instruct" 后缀：
#   "Qwen/Qwen3-8B"      → chat 模型（本教程使用）
#   "Qwen/Qwen3-8B-Base" → 基座，只做续写不按对话回答
model_id = "Qwen/Qwen3-8B"

# 4-bit 量化配置（原理与手算示例见「关于 4-bit 量化」一节）
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # 启用 4-bit 量化
    bnb_4bit_compute_dtype=torch.float16,    # 反量化后矩阵乘法的 dtype（T4 不原生支持 BF16；L4/A100 可改 torch.bfloat16）
    bnb_4bit_quant_type="nf4",               # 量化点采用 NF4（NormalFloat4）
    bnb_4bit_use_double_quant=True,          # 启用双量化，再把每块的 scale 也量化一次
)

# from_pretrained() 自动从 Hugging Face Hub 下载权重并缓存到 ~/.cache/huggingface/
# 下次加载同一模型不会重复下载
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",                # 让 accelerate 自动分配模型层；单卡时等价于全部放到 GPU
    quantization_config=quant_config, # 应用上面定义的 4-bit 量化配置
)

# 加载与模型配对的分词器：负责文本 ↔ token id 的双向转换
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# ============================================================
# Cell 3: 看一眼磁盘上的模型——cache 文件 + model.config
# ============================================================
# 1.1 节讲了 from_pretrained() 把仓库下载到哪、文件是什么格式、
# config.json 长什么样——这里把它们实际打印出来感受一下
import os

# 缓存路径：第一次 from_pretrained 会把仓库下载到这里
# Colab 容器内默认是 /root/.cache/huggingface/hub/
base = os.path.expanduser("~/.cache/huggingface/hub/models--Qwen--Qwen3-8B")
print(f"=== Cache 目录：{base} ===\n")

# 列出全部文件 + 大小，看清 blobs / refs / snapshots 三层关系
# blobs/ 里是真正的字节内容（按 sha 命名），snapshots/ 里是带可读名字的软链
for root, _, files in os.walk(base):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{size_mb:8.1f} MB  {path}")

# 模型类与架构超参——对应 config.json
# Qwen3ForCausalLM 这个具体类是从 config.json 里 model_type=qwen3 推断出来的
print(f"\n=== model.__class__.__name__ ===")
print(model.__class__.__name__)

print(f"\n=== model.config ===")
print(model.config)

In [ ]:
# ============================================================
# Cell 4: 观察 temperature 对 softmax 分布的影响（最小数值例子）
# ============================================================
# 词表只有 4 个 token 的玩具情形，logits = [2.0, 1.0, 0.5, 0.1]
# 看看不同 temperature 下，softmax 概率分布如何变化
import torch

logits = torch.tensor([2.0, 1.0, 0.5, 0.1])

print(f"原始 logits: {logits.tolist()}\n")
for T in [0.1, 0.7, 1.0, 2.0, 5.0]:
    probs = torch.softmax(logits / T, dim=-1)
    formatted = [f"{p:.4f}" for p in probs.tolist()]
    print(f"T = {T:>4}: P = {formatted}")

In [ ]:
# ============================================================
# Cell 5: 定义 chat() 辅助函数，把对话生成的样板代码封装起来
# ============================================================
# 后续所有实验都会反复调用 generate()，只是采样参数不同——
# 把不变的部分（构造 messages、apply chat template、移到 GPU、
# 切片解码）封进函数，每个实验 cell 只关心要变的旋钮

# 提前查一次 <|im_end|> 的 token id；对话场景下作为 EOS 之一
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")


def chat(prompt, *, enable_thinking=False, max_new_tokens=200, **gen_kwargs):
    """便捷封装：传入 user prompt，返回模型回答字符串

    gen_kwargs 透传给 model.generate()，可包含：
      do_sample / temperature / top_p / top_k / repetition_penalty / ...
    """
    # 构造对话 messages（本章实验都不带 system prompt，让差异完全来自采样参数）
    messages = [{"role": "user", "content": prompt}]

    # 应用 chat template，得到 input_ids + attention_mask 字典
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,    # 末尾追加 <|im_start|>assistant\n
        enable_thinking=enable_thinking,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    # 推理不算梯度，省显存又加速
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            # 采到通用 EOS 或 <|im_end|> 任一一个就停
            eos_token_id=[tokenizer.eos_token_id, im_end_id],
            # batch=1 也显式传 pad_token_id 一下，避免 transformers 报警告
            pad_token_id=tokenizer.eos_token_id,
            **gen_kwargs,
        )

    # outputs 形状 [1, 输入长度 + 新生成长度]，只取新生成部分
    input_length = inputs["input_ids"].shape[-1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response

In [ ]:
# ============================================================
# Cell 6: 实验 1 —— 贪心解码 vs 多项式采样
# ============================================================
# 同一个 prompt，两种解码策略各跑 3 次，看输出有何不同
prompt = "请用一句话介绍 Python 编程语言。"

print("【贪心解码 do_sample=False】")
for i in range(3):
    print(f"  Run {i+1}: {chat(prompt, do_sample=False)}")

print("\n【多项式采样 do_sample=True, temperature=0.7, top_p=0.8, top_k=20】")
for i in range(3):
    torch.manual_seed(i)  # 每次用不同 seed，让采样路径不同
    print(f"  Run {i+1}: {chat(prompt, do_sample=True, temperature=0.7, top_p=0.8, top_k=20)}")

In [ ]:
# ============================================================
# Cell 7: 实验 2 —— temperature 对输出"发散度"的影响
# ============================================================
# 用一个开放性 prompt（写诗），对比 T=0.3 / 0.7 / 1.5 各跑两次
prompt = "用四行写一首关于秋天的小诗，要押韵。"

for T in [0.3, 0.7, 1.5]:
    print(f"\n=== temperature = {T} ===")
    for i in range(2):
        torch.manual_seed(i)
        print(f"--- Run {i+1} ---")
        print(chat(prompt, do_sample=True, temperature=T, top_p=0.95, top_k=50))

In [ ]:
# ============================================================
# Cell 8: 实验 3 —— top_p 严格 vs 宽松
# ============================================================
# 同一个 prompt，对比 top_p=0.3（严格，只在最稳的几个候选里挑）
# 与 top_p=0.95（宽松，长尾也有机会被采到）
# 故意把 top_k 设成 0（不限），让差异完全来自 top_p
prompt = "推荐一种适合初学者养的宠物，并说明理由。"

for p in [0.3, 0.95]:
    print(f"\n=== top_p = {p} ===")
    for i in range(2):
        torch.manual_seed(i)
        print(f"--- Run {i+1} ---")
        print(chat(prompt, do_sample=True, temperature=0.8, top_p=p, top_k=0))

In [ ]:
# ============================================================
# Cell 9: 实验 4 —— repetition_penalty 抑制重复
# ============================================================
# 故意构造容易让模型循环的输入：列举 + 短句格式
# 对比 repetition_penalty=1.0（无抑制）vs 1.2（强抑制）
prompt = "请列出 5 个学习编程的好处，每条不超过 15 个字。"

for rp in [1.0, 1.2]:
    print(f"\n=== repetition_penalty = {rp} ===")
    torch.manual_seed(0)
    print(chat(
        prompt,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        repetition_penalty=rp,
        max_new_tokens=300,
    ))

In [ ]:
# ============================================================
# Cell 10: 实验 5 —— 思考模式 vs 非思考模式
# ============================================================
# 一道需要列方程的小题，对比两种模式下的输出与正确率
# 思考模式必须 do_sample=True 且 max_new_tokens 给足（≥ 2048）
prompt = (
    "小明买了 3 支铅笔和 2 块橡皮，共 14 元；"
    "他又买了 5 支铅笔和 1 块橡皮，共 19 元。"
    "请问铅笔单价是多少？"
)

print("=== 非思考模式 enable_thinking=False ===")
torch.manual_seed(0)
print(chat(
    prompt,
    enable_thinking=False,
    do_sample=True,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    max_new_tokens=300,
))

print("\n=== 思考模式 enable_thinking=True ===")
torch.manual_seed(0)
print(chat(
    prompt,
    enable_thinking=True,
    do_sample=True,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    max_new_tokens=2048,
))

In [ ]:
# ============================================================
# Cell 11: 实验 6 —— 复现性与随机种子
# ============================================================
# 同样的采样配置，固定 seed 后两次运行是否一致？
prompt = "解释一下什么是递归，并给一个 Python 示例。"

print("=== 同一个 seed, 两次运行 ===")
for i in range(2):
    torch.manual_seed(2024)
    print(f"--- Run {i+1} ---")
    print(chat(prompt, do_sample=True, temperature=0.7, top_p=0.8, top_k=20))

print("\n=== 不设 seed, 两次运行 ===")
for i in range(2):
    print(f"--- Run {i+1} ---")
    print(chat(prompt, do_sample=True, temperature=0.7, top_p=0.8, top_k=20))

print("\n=== 贪心解码 do_sample=False，无需 seed ===")
for i in range(2):
    print(f"--- Run {i+1} ---")
    print(chat(prompt, do_sample=False))